# Native Ads Detection - Classification Demo

This notebook demonstrates how to use the fine-tuned model to classify Indonesian news articles as native ads or pure news.

## Setup

In [ ]:
import sys
sys.path.append('../langchain-refactor')

from chains.full_pipeline_chain import FullPipelineChain
import json
from datetime import datetime

## Initialize Pipeline

Choose your model provider:
- `local`: Fine-tuned Qwen 2.5 14B model
- `openrouter`: GPT-4o-mini via OpenRouter
- `openai`: GPT-3.5/4 via OpenAI

In [ ]:
# Configuration
PROVIDER = "local"  # Change to 'openrouter' or 'openai' if needed
MODEL_PATH = "../models/native-ads-qwen14b_merged_16bit"  # For local provider
API_KEY = None  # Set if using openrouter/openai

# Initialize pipeline
pipeline = FullPipelineChain(
    model_name=MODEL_PATH if PROVIDER == "local" else "gpt-4o-mini",
    provider=PROVIDER,
    api_key=API_KEY
)

print("✅ Pipeline initialized!")

## Test with Sample URLs

### Example 1: Native Ads (OPPO Product)

In [ ]:
url1 = "https://tekno.sindonews.com/read/542366/776/begini-cara-mengoptimalkan-fitur-di-oppo-enco-buds-1631779764"

result1 = pipeline.run(url1)

print("="*80)
print("CLASSIFICATION RESULT")
print("="*80)
print(f"URL: {url1}")
print(f"Title: {result1.get('title', 'N/A')}")
print(f"\nLabel: {result1['classification']['label']}")
print(f"Confidence: {result1['classification']['confidence']:.2f}")
print(f"\nReasoning:\n{result1['classification']['reasoning']}")
print("="*80)

### Example 2: Pure News (Critical Article)

In [ ]:
url2 = "https://tekno.sindonews.com/read/474336/123/example-pure-news-article"

result2 = pipeline.run(url2)

print("="*80)
print("CLASSIFICATION RESULT")
print("="*80)
print(f"URL: {url2}")
print(f"Title: {result2.get('title', 'N/A')}")
print(f"\nLabel: {result2['classification']['label']}")
print(f"Confidence: {result2['classification']['confidence']:.2f}")
print(f"\nReasoning:\n{result2['classification']['reasoning']}")
print("="*80)

## Batch Classification

Process multiple URLs at once

In [ ]:
urls = [
    "https://tekno.sindonews.com/read/542366/776/begini-cara-mengoptimalkan-fitur-di-oppo-enco-buds-1631779764",
    "https://tekno.sindonews.com/read/447848/776/inovasi-hp-kamera-terbaik-vivo-pada-7-tahun-kiprahnya-di-indonesia-1622988418",
    # Add more URLs here
]

results = pipeline.run_batch(urls)

# Summary
native_ads_count = sum(1 for r in results if r['classification']['label'] == 'native ads')
berita_murni_count = len(results) - native_ads_count

print(f"\n📊 Batch Classification Summary:")
print(f"   Total URLs: {len(results)}")
print(f"   Native Ads: {native_ads_count}")
print(f"   Berita Murni: {berita_murni_count}")

# Detailed results
for i, result in enumerate(results, 1):
    print(f"\n{i}. {result.get('title', 'N/A')[:50]}...")
    print(f"   Label: {result['classification']['label']} (confidence: {result['classification']['confidence']:.2f})")

## Save Results

Export results to JSON for further analysis

In [ ]:
# Save to JSON
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = f"../results/batch_classification_{timestamp}.json"

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"✅ Results saved to: {output_file}")

## Custom URL Classification

Try with your own URL

In [ ]:
# Enter your URL here
custom_url = input("Enter URL to classify: ")

result = pipeline.run(custom_url)

print("\n" + "="*80)
print("CLASSIFICATION RESULT")
print("="*80)
print(f"URL: {custom_url}")
print(f"Title: {result.get('title', 'N/A')}")
print(f"\nLabel: {result['classification']['label']}")
print(f"Confidence: {result['classification']['confidence']:.2f}")
print(f"\nReasoning:\n{result['classification']['reasoning']}")

if result.get('explanation'):
    print(f"\nDetailed Explanation:\n{result['explanation']}")

print("="*80)